# TC-8 — PWR Service Water Check Valve: Repeated Failures → Systemic Organizational Root Cause

**Plant:** Unit 1, Facility B (PWR) | **System:** Service Water (SW) | **Event:** EVT-U1B-2025-0312

Check valve `CHK-SW-HX-07A` (model `GWB-250-SS-316`, EDG HX-07A Train A supply header) is found
leaking through at 0.42 gpm during the quarterly flow balance surveillance — **the third leakthrough
failure of the same valve model in 18 months**.

The shift supervisor's initial hypothesis: *"Valve wear, normal service life. Replace with like-for-like."*

This is the wrong answer. DACKAR's structured investigation reveals five causal contributors spanning
four categories (A, I, J, K, L) across three causal depths (proximate / contributing / root cause).

**Key features demonstrated:**
- Multi-category candidate generation (A, I, J, K, L) — first use of Category L (systemic/organizational) in the suite
- Recurrence detection across 3 historical CAP entries (TSKR episode_recurrence patterns)
- Fleet OE integration (INPO IRIS report as Category K evidence)
- PM frequency nonconformance detected via KG PM task node interval mismatch (Category J)
- Conflicting evidence handling: teardown report contradicts vendor batch hypothesis; resolved by NER lot-number cross-reference
- Intentional residual ambiguity: 2 open items requiring analyst determination

In [9]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR   = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR    = NOTEBOOK_ROOT / "rca_runs_case_008" / "v32"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [
    os.path.abspath(os.path.join(os.getcwd(), "..", ".."))       ,
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")) ,
    os.path.abspath(os.path.join(os.getcwd(), "..", "shared"))   ,
]:
    if p not in sys.path:
        sys.path.insert(0, p)

from run_helpers import build_fixture_orchestrator, load_fixtures, run_rca, print_block, summarise_result
from assertion_helpers import (
    assert_candidate_count,
    assert_candidate_present,
    assert_data_coverage_status,
    assert_depth_complete,
    assert_unresolved_gaps_at_least,
    run_assertion_table,
)
print("Imports OK. Fixture dir:", FIXTURE_DIR)

Imports OK. Fixture dir: /Users/mandd/projects/DACKAR/src/dackar/RCA/tests/test_case_8/fixtures


In [10]:
fixtures = load_fixtures(FIXTURE_DIR)
print("Fixtures loaded:")
for k, v in fixtures.items():
    status = "present" if v is not None else "absent"
    print(f"  {k}: {status}")

Fixtures loaded:
  event: present
  telemetry_summary: present
  kg_context: present
  operational_context: present
  pm_compliance: present
  tskr_patterns: present
  evidence_bundle: present
  soe_log: present
  alarm_log: absent
  protection_logic_context: absent
  configuration_change_records: absent
  environmental_monitoring: absent
  vendor_supply_chain_records: absent
  training_records: absent


In [11]:
orc = build_fixture_orchestrator(OUTPUT_DIR, top_k_candidates=6, enable_ishikawa=True)
result = run_rca(orc, fixtures)
print("Run complete. Top-level keys:", list(result.keys()))

Run complete. Top-level keys: ['run_context', 'pm_compliance', 'kg_context', 'signal_evidence', 'tskr_patterns', 'causality_candidates', 'causality_candidates_pre_refine', 'evidence_bundle', 'ishikawa_matrix', 'barrier_analysis', 'reentry_execution', 'cmms_context', 'rca_card', 'input_validation', 'output_validation', 'run_manifest']


### Pre-Refinement Candidate Scores

Five candidates are generated across four causal categories. At pre-refine, all candidates share
the same document-based evidence prior (all KG documents are visible to all failure modes).
Category J > Category A composite by design — governance weight is dominant for Category J
and the pm_compliance interval nonconformance drives a high governance score.

Note: scores across different categories are **not directly comparable** — they reflect different
weight profiles.

In [12]:
print("=== PRE-REFINEMENT CANDIDATES ===")
pre_cands = (result.get("causality_candidates_pre_refine") or {}).get("candidates") or []
print(f"Retained: {len(pre_cands)}")
for c in sorted(pre_cands, key=lambda x: -float(x.get("composite_score", 0) or 0)):
    s = c.get("scores") or {}
    print(
        f"  [{c.get('primary_causal_category','?')}] "
        f"{c.get('failure_mode_id','?'):35s} "
        f"composite={float(c.get('composite_score', 0) or 0):.3f}  "
        f"E={float(s.get('evidence', 0) or 0):.3f}  "
        f"G={float(s.get('governance', 0) or 0):.3f}"
    )

=== PRE-REFINEMENT CANDIDATES ===
Retained: 6
  [A] FM-CHK-SEAT-EROSION                 composite=0.656  E=0.412  G=0.900
  [A] FM-CHK-DISC-DAMAGE                  composite=0.610  E=0.412  G=0.750
  [J] FM-PM-FREQ-NONCONF                  composite=0.595  E=0.412  G=0.900
  [I] FM-PM-CONFIG-CONTROL-GAP            composite=0.564  E=0.412  G=0.750
  [K] FM-VENDOR-BATCH-TRACEABILITY        composite=0.560  E=0.412  G=0.750
  [L] FM-OE-SCREENING-MISS                composite=0.537  E=0.412  G=0.750


### Post-Refinement Candidate Scores

After evidence refinement, all composite scores decrease relative to pre-refine due to the quality
multiplier (~0.68) applied in uncertainty propagation — this is expected behavior.

Relative ranking after refinement:
- **FM-PM-FREQ-NONCONF (J):** highest composite — Category J's dominant evidence weight (0.55) and
  governance weight (0.30) reward the extremely well-documented PM interval deviation (factual CR/WO/SPEC records, zero conjecture)
- **FM-CHK-SEAT-EROSION (A):** second — three supporting snippets but Category A's lower evidence weight (0.20) limits uplift
- **FM-PM-CONFIG-CONTROL-GAP (I):** third — contributing organizational cause, moderate governance
- **FM-VENDOR-BATCH-TRACEABILITY (K):** fourth — retained despite contradicting evidence (modulated contradiction score)
- **FM-OE-SCREENING-MISS (L):** fifth — high conjecture fraction (0.38) from the OE screening log caps confidence

**Causal depth is assigned by category letter, not score rank.** FM-OE-SCREENING-MISS (L) is the
root-cause-depth candidate regardless of its composite score.

In [13]:
print("=== POST-REFINEMENT CANDIDATES ===")
post_cands = (result.get("causality_candidates") or {}).get("candidates") or []
print(f"Retained: {len(post_cands)}")
for c in sorted(post_cands, key=lambda x: -float(x.get("composite_score", 0) or 0)):
    s = c.get("scores") or {}
    print(
        f"  [{c.get('primary_causal_category','?')}] "
        f"{c.get('failure_mode_id','?'):35s} "
        f"composite={float(c.get('composite_score', 0) or 0):.3f}  "
        f"E={float(s.get('evidence', 0) or 0):.3f}  "
        f"G={float(s.get('governance', 0) or 0):.3f}  "
        f"posture={c.get('evidence_posture','?')}"
    )

=== POST-REFINEMENT CANDIDATES ===
Retained: 5
  [J] FM-PM-FREQ-NONCONF                  composite=0.456  E=0.561  G=0.900  posture=supported
  [A] FM-CHK-SEAT-EROSION                 composite=0.454  E=0.493  G=0.900  posture=supported
  [I] FM-PM-CONFIG-CONTROL-GAP            composite=0.391  E=0.448  G=0.750  posture=supported
  [K] FM-VENDOR-BATCH-TRACEABILITY        composite=0.360  E=0.362  G=0.750  posture=weak
  [L] FM-OE-SCREENING-MISS                composite=0.347  E=0.376  G=0.750  posture=weak


### Conflicting Evidence Resolution — The Lot Number Moment

**TD-REPORT-2025-0312** (teardown inspection) states seat insert OD = 2.247 in, within acceptance
criteria 2.240–2.260 in. Initial role: **contradicting** FM-VENDOR-BATCH-TRACEABILITY.

NER cross-reference resolves this: lot GWB-2020-L07 falls within the affected range GWB-2020-L05
through L09; IRIS-OE specifies fatigue failure after 12–24 months, not at initial inspection.
The contradiction weight is **modulated** (not zeroed) — role label stays 'contradicting'.

The modulation is encoded in the evidence bundle as `best_contradiction_score=0.20` rather than 1.0.
FM-VENDOR-BATCH-TRACEABILITY is **retained** in post-refine candidates despite the contradiction;
without lot-number cross-reference the contradiction would be scored at full weight and would produce
a lower evidence score. Overall composite scores fall post-refine (quality multiplier applied to all
candidates), but the modulation keeps K retained above the evidence floor.

In [14]:
def get_by_fm_id(cands, fm_id):
    return next((c for c in cands if c.get("failure_mode_id") == fm_id), None)

pre_batch  = get_by_fm_id(pre_cands,  "FM-VENDOR-BATCH-TRACEABILITY")
post_batch = get_by_fm_id(post_cands, "FM-VENDOR-BATCH-TRACEABILITY")

if pre_batch and post_batch:
    pre_e  = float((pre_batch.get("scores")  or {}).get("evidence", 0) or 0)
    post_e = float((post_batch.get("scores") or {}).get("evidence", 0) or 0)
    pre_c  = float(pre_batch.get("composite_score", 0) or 0)
    post_c = float(post_batch.get("composite_score", 0) or 0)
    print("FM-VENDOR-BATCH-TRACEABILITY evidence scores:")
    print(f"  Pre-refine   E={pre_e:.3f}  composite={pre_c:.3f}")
    print(f"  Post-refine  E={post_e:.3f}  composite={post_c:.3f}")
    print(f"  Score increase: {'YES' if post_c > pre_c else 'NO'} (delta={post_c - pre_c:+.3f})")
else:
    print("FM-VENDOR-BATCH-TRACEABILITY not found in candidates")

# Show the contradicting evidence snippet from the evidence bundle
eb = result.get("evidence_bundle") or {}
for snip in eb.get("results", []):
    meta = snip.get("metadata") or {}
    if meta.get("support_role") == "contradicting":
        print(f"\nContradicting document: {snip['doc_id']}")
        print(f"  support_role: {meta.get('support_role')}  score: {snip.get('score')}")
        print(f"  Snippet: {snip['snippet'][:200]}...")

FM-VENDOR-BATCH-TRACEABILITY evidence scores:
  Pre-refine   E=0.412  composite=0.560
  Post-refine  E=0.362  composite=0.360
  Score increase: NO (delta=-0.200)

Contradicting document: TD-REPORT-2025-0312
  support_role: contradicting  score: 0.82
  Snippet: Teardown inspection of valve CHK-SW-HX-07A (removed 2025-03-12). Lot number: GWB-2020-L07. Seat insert OD: 2.247 in — within acceptance criteria 2.240-2.260 in. Finding: poppet seat shows wear pattern...


In [15]:
summarise_result(result)


  RCA RUN SUMMARY
  run_id          : d3cfd493-2e26-44b8-b082-6b50ed6f35b7
  event_id        : n/a
  decision_status : n/a
  fallback_used   : n/a

  PRIMARY HYPOTHESIS
    cause_label   : Inspection/testing program inadequacy — PM frequency nonconformance with vendor specification
    causal_cat    : n/a
    composite_score: 0.456494

  CANDIDATES
    retained      : 5
    [1] FM-PM-FREQ-NONCONF             composite=0.456  gates=PASS
    [2] FM-CHK-SEAT-EROSION            composite=0.454  gates=PASS
    [3] FM-PM-CONFIG-CONTROL-GAP       composite=0.391  gates=PASS
    [4] FM-VENDOR-BATCH-TRACEABILITY   composite=0.360  gates=PASS
    [5] FM-OE-SCREENING-MISS           composite=0.347  gates=PASS

  DATA COVERAGE
    kg_context                         : complete
    chroma_corpus                      : complete
    upstream_anomaly_inputs            : complete
    telemetry_detail                   : complete
    soe_log                            : complete
    alarm_log           

### Assertions

In [16]:
def _get_by_fm(cands, fm_id):
    return next((c for c in cands if c.get("failure_mode_id") == fm_id), None)

def _categories_present(result, expected):
    cands = (result.get("causality_candidates") or {}).get("candidates") or []
    found = {c.get("primary_causal_category") for c in cands}
    missing = [cat for cat in expected if cat not in found]
    assert not missing, f"Missing causal categories: {missing} (found: {found})"

def _check_highest_composite(result, fm_id):
    cands = (result.get("causality_candidates") or {}).get("candidates") or []
    target = _get_by_fm(cands, fm_id)
    assert target is not None, f"{fm_id} not found in candidates"
    max_score = max(float(c.get("composite_score", 0) or 0) for c in cands)
    my_score  = float(target.get("composite_score", 0) or 0)
    assert my_score >= max_score - 1e-6, (
        f"{fm_id} composite={my_score:.3f} is not the highest (max={max_score:.3f})"
    )

def _check_category(result, fm_id, expected_cat):
    cands = (result.get("causality_candidates") or {}).get("candidates") or []
    target = _get_by_fm(cands, fm_id)
    assert target is not None, f"{fm_id} not found in candidates"
    actual = target.get("primary_causal_category")
    assert actual == expected_cat, f"{fm_id} category={actual}, expected {expected_cat}"

def _check_contradiction_modulated(result, fm_id):
    """Assert fm_id is retained post-refine despite contradicting evidence,
    and its contradiction score in the evidence bundle is modulated (< 0.5)."""
    post_cands = (result.get("causality_candidates") or {}).get("candidates") or []
    assert any(c.get("failure_mode_id") == fm_id for c in post_cands), (
        f"{fm_id} was filtered out despite contradiction modulation"
    )
    eb = result.get("evidence_bundle") or {}
    ces = eb.get("candidate_evidence_summary") or []
    entry = next((e for e in ces if e.get("candidate_id") == f"FM::{fm_id}"), None)
    assert entry is not None, f"No evidence summary entry for FM::{fm_id}"
    contra = float(entry.get("best_contradiction_score", 1.0) or 1.0)
    assert contra < 0.5, (
        f"{fm_id} best_contradiction_score={contra:.2f} is not modulated — expected < 0.5 "
        f"(lot-number cross-reference should reduce contradiction weight)"
    )

def _check_pm_compliance_failed(result):
    """Assert pm_compliance fixture was received and records at least one failed check."""
    pm = result.get("pm_compliance") or {}
    assert pm, "pm_compliance artifact is absent from result"
    failed = (pm.get("summary") or {}).get("failed", 0)
    assert failed >= 1, (
        f"pm_compliance.summary.failed={failed} — expected >= 1 "
        f"(interval nonconformance vs vendor spec)"
    )

def _check_contradicting_doc(result, doc_id):
    eb = result.get("evidence_bundle") or {}
    for snip in eb.get("results", []):
        meta = snip.get("metadata") or {}
        if snip.get("doc_id") == doc_id and meta.get("support_role") == "contradicting":
            return
    raise AssertionError(
        f"No snippet with doc_id='{doc_id}' and metadata.support_role='contradicting' found"
    )

ASSERTIONS = [
    {
        "id": "A8-1",
        "desc": "Five candidates retained in post-refine",
        "fn": lambda r: assert_candidate_count(r, 5),
    },
    {
        "id": "A8-2",
        "desc": "Category span A/I/J/K/L all present",
        "fn": lambda r: _categories_present(r, ["A", "I", "J", "K", "L"]),
    },
    {
        "id": "A8-3",
        "desc": "FM-PM-FREQ-NONCONF has highest post-refine composite",
        "fn": lambda r: _check_highest_composite(r, "FM-PM-FREQ-NONCONF"),
    },
    {
        "id": "A8-4",
        "desc": "FM-CHK-SEAT-EROSION is Category A (proximate)",
        "fn": lambda r: _check_category(r, "FM-CHK-SEAT-EROSION", "A"),
    },
    {
        "id": "A8-5",
        "desc": "FM-OE-SCREENING-MISS is Category L (root cause)",
        "fn": lambda r: _check_category(r, "FM-OE-SCREENING-MISS", "L"),
    },
    {
        "id": "A8-6",
        "desc": "FM-VENDOR-BATCH-TRACEABILITY retained despite contradicting evidence (lot-number modulation)",
        "fn": lambda r: _check_contradiction_modulated(r, "FM-VENDOR-BATCH-TRACEABILITY"),
    },
    {
        "id": "A8-7",
        "desc": "TD-REPORT-2025-0312 has contradicting role in evidence bundle",
        "fn": lambda r: _check_contradicting_doc(r, "TD-REPORT-2025-0312"),
    },
    {
        "id": "A8-8",
        "desc": "PM compliance fixture present with >= 1 failed check (interval nonconformance detected)",
        "fn": lambda r: _check_pm_compliance_failed(r),
    },
    {
        "id": "A8-9",
        "desc": "Causal depth complete (proximate + contributing + root all present)",
        "fn": lambda r: assert_depth_complete(r, True),
    },
    {
        "id": "A8-10",
        "desc": "rca_card has >= 2 open items (intentional residual ambiguity)",
        "fn": lambda r: assert_unresolved_gaps_at_least(r, 2),
    },
]

run_assertion_table(result, ASSERTIONS, label="TC-8 Assertions")


--- TC-8 Assertions ---
  pass  len(candidates) == 5  (got 5)
  PASS  [A8-1] Five candidates retained in post-refine
  PASS  [A8-2] Category span A/I/J/K/L all present
  PASS  [A8-3] FM-PM-FREQ-NONCONF has highest post-refine composite
  PASS  [A8-4] FM-CHK-SEAT-EROSION is Category A (proximate)
  PASS  [A8-5] FM-OE-SCREENING-MISS is Category L (root cause)
  PASS  [A8-6] FM-VENDOR-BATCH-TRACEABILITY retained despite contradicting evidence (lot-number modulation)
  PASS  [A8-7] TD-REPORT-2025-0312 has contradicting role in evidence bundle
  PASS  [A8-8] PM compliance fixture present with >= 1 failed check (interval nonconformance detected)
  pass  executive_summary.causal_depth_summary.depth_complete == True  (got 'True')
  PASS  [A8-9] Causal depth complete (proximate + contributing + root all present)
  pass  executive_summary.unresolved_gaps count >= 2  (got 2)
  PASS  [A8-10] rca_card has >= 2 open items (intentional residual ambiguity)
  All 10 assertion(s) passed.



### What Did the System Do?

**A8-1 — Five candidates generated:** Candidates span all five categories including Category L
(systemic/organizational) — the first test case in the suite to exercise Category L as a ranked candidate.

**A8-2 — Category span A/I/J/K/L:** Proximate (A), contributing (I, J, K), and root cause (L) causal
depths all represented in a single run.

**A8-3 — FM-PM-FREQ-NONCONF highest composite:** Category J's weight profile (w_E=0.55, w_G=0.30) gives
dominant weight to documentary and governance evidence. The PM interval deviation is extremely well
documented (two CRs, the PM revision WO, and the vendor specification — all factual records with zero
conjecture). This lifts J above FM-CHK-SEAT-EROSION despite Category A's structural weight advantage.
Scores are intra-category composites under different weight profiles and are not directly comparable across categories.

**A8-4 / A8-5 — Causal depth correctly assigned:** Depth is determined by category letter, not score
rank. FM-OE-SCREENING-MISS (L) is root-cause depth even if it scores lower than FM-CHK-SEAT-EROSION.

**A8-6 — Vendor batch contradiction modulated:** The NER lot-number cross-reference encodes
`best_contradiction_score=0.20` (not 1.0) in the evidence bundle. FM-VENDOR-BATCH-TRACEABILITY is
retained in post-refine candidates despite the contradicting teardown report. Role label stays
'contradicting' — the resolution is circumstantial, not definitive. All composite scores decrease
post-refine due to the quality multiplier; this is expected behavior.

**A8-7 — Contradicting evidence retained:** The teardown report keeps its 'contradicting' role in the
evidence bundle. The system does not silently reclassify documents to make the hypothesis look cleaner.

**A8-8 — PM compliance detected:** The pm_compliance fixture records 2 failed checks (PM-CHK-SW-INTERVAL-NONCONFORMANCE
and PM-CHK-SW-07B-CROSS-TRAIN), confirming the interval nonconformance spans both trains. The KG PM task
node carries both `plant_interval_months=18` and `vendor_spec_interval_months=12`, enabling Category J
candidate generation.

**A8-9 — Causal depth complete:** Proximate (A), contributing (I, J, K), and root cause (L) all
present. `depth_complete=True` in the executive summary.

**A8-10 — Open items:** Two unresolved gaps remain at card issuance — contradicting evidence requiring
analyst resolution, and sensitivity-table ranking instability if missing data sources become available.
These are features, not failures. The system surfaces ambiguity and forces the analyst to resolve it.